In [1]:
import ast

# Function to convert code string to AST
def code_to_ast(code_string):
	try:
		return ast.parse(code_string)
	except SyntaxError:
		return None # some code snippets are not valid (python 2 instead of python 3)

In [2]:
# Linearizing the AST (with SBT, Structure Based Traversal)
def linearize_ast(node, tokens):
	if node is None:
		return

	node_type = type(node).__name__
	
	# 1. APRIAMO il nodo
	tokens.append(f"({node_type}")

	# 2. Arricchimento (facoltativo ma consigliato per la ROUGE)
	if isinstance(node, ast.FunctionDef):
		tokens.append(f"FUNC_{node.name}")
	elif isinstance(node, ast.arg):
		tokens.append(f"ARG_{node.arg}")
	elif isinstance(node, ast.NamedExpr):
		tokens.append(f"VAR:{node.id}")

	# 3. Visita i figli
	for child in ast.iter_child_nodes(node):
		linearize_ast(child, tokens)

	# 4. CHIUDIAMO il nodo
	tokens.append(f"){node_type}")

In [3]:
linearized_tree = []
linearize_ast(code_to_ast("def add(a, b): return a + b + 3"), linearized_tree)
print(linearized_tree)

['(Module', '(FunctionDef', 'FUNC_add', '(arguments', '(arg', 'ARG_a', ')arg', '(arg', 'ARG_b', ')arg', ')arguments', '(Return', '(BinOp', '(BinOp', '(Name', '(Load', ')Load', ')Name', '(Add', ')Add', '(Name', '(Load', ')Load', ')Name', ')BinOp', '(Add', ')Add', '(Constant', ')Constant', ')BinOp', ')Return', ')FunctionDef', ')Module']


Let's now build a dictionary with Corpora, exactly as we have done before.

In [4]:
from datasets import load_dataset

train_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/train.jsonl",
	split = "train")
valid_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/valid.jsonl",
	split = "train")
test_dataset = load_dataset(
	"json",
	data_files = "./../../data/raw/dataset/python/test.jsonl",
	split = "train")

c:\Users\Sean Andreini\Desktop\Unifi\Machine Learning for Software Analysis\code-summarization-mlsa-project\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
train_linearized_trees = []
train_valid_indices = []
for i, code in enumerate(train_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		train_linearized_trees.append(linearized_tree)
		train_valid_indices.append(i)

valid_linearized_trees = []
valid_valid_indices = []
for i, code in enumerate(valid_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		valid_linearized_trees.append(linearized_tree)
		valid_valid_indices.append(i)

test_linearized_trees = []
test_valid_indices = []
for i, code in enumerate(test_dataset['code']):
	linearized_tree = []
	tree = code_to_ast(code)
	if(tree is not None):
		linearize_ast(tree, linearized_tree)
		test_linearized_trees.append(linearized_tree)
		test_valid_indices.append(i)

print(train_linearized_trees[0])

['(Module', '(FunctionDef', 'FUNC_split_phylogeny', '(arguments', '(arg', 'ARG_p', ')arg', '(arg', 'ARG_level', ')arg', '(Constant', ')Constant', ')arguments', '(Expr', '(Constant', ')Constant', ')Expr', '(Assign', '(Name', '(Store', ')Store', ')Name', '(BinOp', '(Name', '(Load', ')Load', ')Name', '(Add', ')Add', '(Constant', ')Constant', ')BinOp', ')Assign', '(Assign', '(Name', '(Store', ')Store', ')Name', '(Call', '(Attribute', '(Name', '(Load', ')Load', ')Name', '(Load', ')Load', ')Attribute', '(Name', '(Load', ')Load', ')Name', ')Call', ')Assign', '(Return', '(BinOp', '(BinOp', '(Subscript', '(Name', '(Load', ')Load', ')Name', '(Constant', ')Constant', '(Load', ')Load', ')Subscript', '(Add', ')Add', '(Name', '(Load', ')Load', ')Name', ')BinOp', '(Add', ')Add', '(Subscript', '(Call', '(Attribute', '(Subscript', '(Name', '(Load', ')Load', ')Name', '(Constant', ')Constant', '(Load', ')Load', ')Subscript', '(Load', ')Load', ')Attribute', '(Constant', ')Constant', ')Call', '(Constant', 

In [6]:
from gensim import corpora
code_dictionary = corpora.Dictionary(train_linearized_trees)
code_dictionary.filter_extremes(no_below=5, no_above=0.5)
special_tokens = {'[UNK]': 0, '[PAD]': 1, '[BOS]': 2, '[EOS]': 3}
code_dictionary.patch_with_special_tokens(special_tokens)

In [8]:
code_dictionary.token2id

{'(Add': 13684,
 '(BinOp': 13685,
 '(Subscript': 13686,
 ')Add': 13687,
 ')BinOp': 4,
 ')Subscript': 5,
 'ARG_level': 6,
 'ARG_p': 7,
 '(Compare': 8,
 '(Eq': 9,
 '(ExceptHandler': 10,
 '(Not': 11,
 '(Try': 12,
 '(UnaryOp': 13,
 ')Compare': 14,
 ')Eq': 15,
 ')ExceptHandler': 16,
 ')Not': 17,
 ')Try': 18,
 ')UnaryOp': 19,
 'ARG_d': 20,
 'FUNC_ensure_dir': 21,
 '(Raise': 22,
 ')Raise': 23,
 'ARG_mode': 24,
 '(And': 25,
 '(Assert': 26,
 '(BoolOp': 27,
 '(Continue': 28,
 '(Dict': 29,
 '(For': 30,
 '(Gt': 31,
 '(In': 32,
 '(Is': 33,
 '(ListComp': 34,
 '(NotIn': 35,
 '(Tuple': 36,
 '(comprehension': 37,
 ')And': 38,
 ')Assert': 39,
 ')BoolOp': 40,
 ')Continue': 41,
 ')Dict': 42,
 ')For': 43,
 ')Gt': 44,
 ')In': 45,
 ')Is': 46,
 ')ListComp': 47,
 ')NotIn': 48,
 ')Tuple': 49,
 ')comprehension': 50,
 'ARG_categories': 51,
 'ARG_header': 52,
 '(List': 53,
 '(With': 54,
 '(withitem': 55,
 ')List': 56,
 ')With': 57,
 ')withitem': 58,
 '(Break': 59,
 '(Slice': 60,
 '(USub': 61,
 ')Break': 62,
 ')Sli

In [7]:
len(code_dictionary)

13688

We can see that we went from a code_dictionary of over 1 million of tokens (see notebook 02) to just under 14k. This can help speed up the training and give some more accurate results. So, let's try this by just copy-pasting the model from notebook 03. We're gonna use the same vocab for docstrings. We're gonna save the processed dataset and test it on the next notebook.

In [9]:
from datasets import load_from_disk
old_dataset = load_from_disk("../../data/processed/notebooks/tokenized_codexglue")

In [10]:
train_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in train_linearized_trees
]

valid_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in valid_linearized_trees
]

test_input_ids = [
	[code_dictionary.token2id.get(token, code_dictionary.token2id['[UNK]']) for token in tree]
	for tree in test_linearized_trees
]

In [11]:
from datasets import Dataset, DatasetDict

processed_datasets = DatasetDict({
	'train': Dataset.from_dict({
		'input_ids': train_input_ids,
		'labels': [old_dataset['train']['labels'][i] for i in train_valid_indices]
	}),
	'valid': Dataset.from_dict({
		'input_ids': valid_input_ids,
		'labels': [old_dataset['valid']['labels'][i] for i in valid_valid_indices]
	}),
	'test': Dataset.from_dict({
		'input_ids': test_input_ids,
		'labels': [old_dataset['test']['labels'][i] for i in test_valid_indices]
	})
})

We're gonna save a copy of our docstring dictionary, so as to have everything organized.

In [12]:
docstring_dictionary = corpora.Dictionary.load('./../../data/processed/notebooks/tokenized_codexglue/docstring_dictionary.pt')

In [13]:
processed_datasets.save_to_disk('./../../data/processed/notebooks/ast/')
code_dictionary.save('./../../data/processed/notebooks/ast/code_dictionary.pt')
docstring_dictionary.save('./../../data/processed/notebooks/ast/docstring_dictionary.pt')

Saving the dataset (1/1 shards): 100%|██████████| 14761/14761 [00:00<00:00, 1051961.15 examples/s]
